In [ ]:
import os
import nibabel as nib
import numpy as np

# === PARAMÈTRES À MODIFIER SI BESOIN ===
input_dir = "/home/amenacer/Stage/data 4D/4D/3D+t_Coh2_baseline-Nifti"
output_dir = "/home/amenacer/Stage/data 4D/4D/slices_3DplusT"
os.makedirs(output_dir, exist_ok=True)

# === TRAITEMENT DE CHAQUE FICHIER NIFTI 4D ===
for filename in os.listdir(input_dir):
    if filename.endswith(".nii") or filename.endswith(".nii.gz"):
        filepath = os.path.join(input_dir, filename)
        print(f"📂 Lecture : {filename}")

        # Lecture du fichier NIfTI
        img = nib.load(filepath)
        data = img.get_fdata()
        affine = img.affine

        if data.ndim != 4:
            print(f"⛔ Fichier ignoré (non 4D): {filename}")
            continue

        x, y, z, t = data.shape
        print(f"→ Dimensions : ({x}, {y}, {z}, {t})")

        for z_idx in range(z):
            # Extraire le slice (128, 128, 1, T)
            slice_3dT = data[:, :, z_idx, :].squeeze()  # devient (128, 128, T)

            # Créer une nouvelle image NIfTI
            new_img = nib.Nifti1Image(slice_3dT, affine)

            # Construire un nom de fichier
            base_name = os.path.splitext(os.path.splitext(filename)[0])[0]
            new_filename = f"{base_name}_z{z_idx:02d}_3DplusT.nii.gz"
            output_path = os.path.join(output_dir, new_filename)

            # Sauvegarde
            nib.save(new_img, output_path)
            print(f"✅ Sauvegardé : {new_filename}")

print("\n🎉 Terminé ! Tous les slices ont été extraits.")


In [ ]:
import os
import nibabel as nib
import numpy as np

# Dossier source et destination
input_dir = "/home/amenacer/Stage/data 4D/4D/slices_3DplusT"
output_dir = "/home/amenacer/Stage/data 4D/4D/slices_squeezed"
os.makedirs(output_dir, exist_ok=True)

# Fichiers à traiter
files = [f for f in os.listdir(input_dir) if f.endswith(".nii") or f.endswith(".nii.gz")]

for filename in files:
    filepath = os.path.join(input_dir, filename)
    print(f"📂 Lecture : {filename}")
    
    # Chargement de l'image
    img = nib.load(filepath)
    data = img.get_fdata()
    
    # Vérification des dimensions
    shape = data.shape
    print(f"→ Dimensions originales : {shape}")
    
    # Suppression de la dimension singleton si présente
    squeezed_data = np.squeeze(data)
    
    # Affichage des dimensions squeezées
    print(f"→ Dimensions squeezées : {squeezed_data.shape}")
    
    # Création et sauvegarde de l’image squeezée
    squeezed_img = nib.Nifti1Image(squeezed_data, img.affine, img.header)
    output_path = os.path.join(output_dir, filename.replace(".nii", "_squeezed.nii").replace(".nii.gz", "_squeezed.nii.gz"))
    nib.save(squeezed_img, output_path)
    
    print(f"✅ Sauvegardé : {os.path.basename(output_path)}\n")


In [ ]:
import os

# Dossier contenant les fichiers à renommer
folder = "/home/amenacer/Stage/data 4D/4D/slices_squeezed"

# Parcours des fichiers
for fname in sorted(os.listdir(folder)):
    if fname.endswith(".nii.gz") and "_3DplusT_squeezed_squeezed" in fname:
        # Nouveau nom : on enlève le suffixe et ajoute _0000
        new_fname = fname.replace("_3DplusT_squeezed_squeezed.nii.gz", "_0000.nii.gz")

        # Chemins complets
        old_path = os.path.join(folder, fname)
        new_path = os.path.join(folder, new_fname)

        os.rename(old_path, new_path)
        print(f"✅ {fname} → {new_fname}")
